# 1. Import Dependencies

In [2]:
import cv2
import numpy as np
import os
import json
import shutil

import mediapipe as mp
from glob import glob
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, multilabel_confusion_matrix
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import TensorBoard, ModelCheckpoint, EarlyStopping

# 2. Initialize MediaPipe and Configuration

In [ ]:
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

ACTIONS = np.array([
    'Fine',
    'Go',
    'Help',
    'No',
    'Water',
    'Yes'
])

SEQUENCE_LENGTH = 30
FEATURE_DIM = 258  # pose(132) + left hand(63) + right hand(63)

WLASL_RAW = os.path.join('..', 'data', 'WLASL', 'WLASL_raw')
OWN_RAW = os.path.join('..', 'data', 'WLASL', 'Own_raw')
SEQ_DATA = os.path.join('..', 'data', 'WLASL', 'SequenceData')
os.makedirs(SEQ_DATA, exist_ok=True)

# 3. MediaPipe Detection and Keypoint Extraction Functions

In [4]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results

def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33 * 4)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21 * 3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21 * 3)
    return np.concatenate([pose, lh, rh])

def sample_or_pad_sequence(sequence, target_len=30):
    sequence = np.array(sequence, dtype=np.float32)
    if len(sequence) == 0:
        return np.zeros((target_len, FEATURE_DIM), dtype=np.float32)

    if len(sequence) == target_len:
        return sequence

    if len(sequence) > target_len:
        idx = np.linspace(0, len(sequence) - 1, target_len).astype(int)
        return sequence[idx]

    pad_count = target_len - len(sequence)
    pad = np.repeat(sequence[-1][None, :], pad_count, axis=0)
    return np.vstack([sequence, pad])

# 4. Video Processing Utilities (Load, Sample, Convert to Keypoints)

In [5]:
VIDEO_EXTS = ('*.mp4', '*.avi', '*.mov', '*.mkv')

def list_videos(folder):
    files = []
    for ext in VIDEO_EXTS:
        files.extend(glob(os.path.join(folder, ext)))
    return sorted(files)

def process_video_to_npy(video_path, save_path, holistic):
    cap = cv2.VideoCapture(video_path)
    keypoints_seq = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        _, results = mediapipe_detection(frame, holistic)
        keypoints = extract_keypoints(results)
        keypoints_seq.append(keypoints)

    cap.release()

    sequence = sample_or_pad_sequence(keypoints_seq, SEQUENCE_LENGTH)
    np.save(save_path, sequence)

# 5. Extract Keypoints from WLASL Videos

In [6]:
for action in ACTIONS:
    os.makedirs(os.path.join(SEQ_DATA, action), exist_ok=True)

with mp_holistic.Holistic(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as holistic:

    for action in ACTIONS:
        if action == 'Idle':
            continue  # Idle comes from your own videos only

        class_dir = os.path.join(WLASL_RAW, action)
        if not os.path.exists(class_dir):
            print(f"Missing WLASL folder: {class_dir}")
            continue

        videos = list_videos(class_dir)
        print(action, len(videos))

        for i, video_path in enumerate(videos):
            save_path = os.path.join(SEQ_DATA, action, f"wlasl_{i}.npy")
            process_video_to_npy(video_path, save_path, holistic)

Fine 9
Go 15
Help 14
No 11
Water 9
Yes 12


# 6. Extract Keypoints from Custom Recorded Videos

In [7]:
with mp_holistic.Holistic(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as holistic:

    for action in ACTIONS:
        class_dir = os.path.join(OWN_RAW, action)
        if not os.path.exists(class_dir):
            print(f"Missing OWN folder: {class_dir}")
            continue

        videos = list_videos(class_dir)
        print(action, len(videos))

        for i, video_path in enumerate(videos):
            save_path = os.path.join(SEQ_DATA, action, f"own_{i}.npy")
            process_video_to_npy(video_path, save_path, holistic)

Missing OWN folder: ..\data\Own_raw\Fine
Missing OWN folder: ..\data\Own_raw\Go
Missing OWN folder: ..\data\Own_raw\Help
Missing OWN folder: ..\data\Own_raw\No
Missing OWN folder: ..\data\Own_raw\Water
Missing OWN folder: ..\data\Own_raw\Yes


# 7. Load Dataset and Build Feature/Label Arrays

In [8]:
label_map = {label: num for num, label in enumerate(ACTIONS)}

X, y = [], []

for action in ACTIONS:
    class_dir = os.path.join(SEQ_DATA, action)
    if not os.path.exists(class_dir):
        continue

    files = sorted(glob(os.path.join(class_dir, '*.npy')))
    for f in files:
        seq = np.load(f)
        if seq.shape == (SEQUENCE_LENGTH, FEATURE_DIM):
            X.append(seq)
            y.append(label_map[action])

X = np.array(X, dtype=np.float32)
y = to_categorical(y).astype(int)

print(X.shape, y.shape)

(70, 30, 258) (70, 6)


# 8. Train / Validation / Test Split

In [9]:
y_labels = np.argmax(y, axis=1)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y_labels
)

y_temp_labels = np.argmax(y_temp, axis=1)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp_labels
)

print(X_train.shape, X_val.shape, X_test.shape)

(56, 30, 258) (7, 30, 258) (7, 30, 258)


# 9. Training Callbacks (TensorBoard, Checkpoint, EarlyStopping)

In [10]:
log_dir = os.path.join('..', 'outputs', 'logs')
os.makedirs(log_dir, exist_ok=True)
os.makedirs(os.path.join('..', 'models'), exist_ok=True)

tb_callback = TensorBoard(log_dir=log_dir)

checkpoint = ModelCheckpoint(
    filepath=os.path.join('..', 'models', 'best_wlasl_finetuned.h5'),
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

# 10. Build LSTM Model

In [11]:
model = Sequential()
model.add(LSTM(64, return_sequences=True, activation='relu', input_shape=(SEQUENCE_LENGTH, FEATURE_DIM)))
model.add(Dropout(0.2))
model.add(LSTM(128, return_sequences=True, activation='relu'))
model.add(Dropout(0.2))
model.add(LSTM(64, return_sequences=False, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(ACTIONS.shape[0], activation='softmax'))

model.compile(
    optimizer='Adam',
    loss='categorical_crossentropy',
    metrics=['categorical_accuracy']
)

model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 30, 64)            82688     
                                                                 
 dropout (Dropout)           (None, 30, 64)            0         
                                                                 
 lstm_1 (LSTM)               (None, 30, 128)           98816     
                                                                 
 dropout_1 (Dropout)         (None, 30, 128)           0         
                                                                 
 lstm_2 (LSTM)               (None, 64)                49408     
                                                                 
 dense (Dense)               (None, 64)                4160      
                                                                 
 dense_1 (Dense)             (None, 32)               

# 11. Train Model

In [12]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    callbacks=[tb_callback, checkpoint],
    batch_size=16
)

Epoch 1/100


4/4 [==============================] - ETA: 0s - loss: 1.8103 - categorical_accuracy: 0.1429 
Epoch 1: val_loss improved from inf to 1.75837, saving model to ..\models\best_wlasl_finetuned.h5
4/4 [==============================] - 6s 246ms/step - loss: 1.8103 - categorical_accuracy: 0.1429 - val_loss: 1.7584 - val_categorical_accuracy: 0.1429
Epoch 2/100
4/4 [==============================] - ETA: 0s - loss: 1.7457 - categorical_accuracy: 0.2321
Epoch 2: val_loss improved from 1.75837 to 1.56210, saving model to ..\models\best_wlasl_finetuned.h5


c:\Users\Joe\OneDrive\Desktop\signmatic-asl-realtime-recognition\.venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


4/4 [==============================] - 0s 51ms/step - loss: 1.7457 - categorical_accuracy: 0.2321 - val_loss: 1.5621 - val_categorical_accuracy: 0.2857
Epoch 3/100
3/4 [=====================>........] - ETA: 0s - loss: 1.6219 - categorical_accuracy: 0.2917
Epoch 3: val_loss did not improve from 1.56210
4/4 [==============================] - 0s 41ms/step - loss: 1.7995 - categorical_accuracy: 0.2679 - val_loss: 1.5647 - val_categorical_accuracy: 0.2857
Epoch 4/100
3/4 [=====================>........] - ETA: 0s - loss: 1.7388 - categorical_accuracy: 0.2083
Epoch 4: val_loss did not improve from 1.56210
4/4 [==============================] - 0s 37ms/step - loss: 1.7416 - categorical_accuracy: 0.2321 - val_loss: 1.7704 - val_categorical_accuracy: 0.2857
Epoch 5/100
4/4 [==============================] - ETA: 0s - loss: 1.7591 - categorical_accuracy: 0.2857
Epoch 5: val_loss did not improve from 1.56210
4/4 [==============================] - 0s 36ms/step - loss: 1.7591 - categorical_accurac

# 12. Evaluate Model Performance

In [13]:
best_model = load_model(os.path.join('..', 'models', 'best_wlasl_finetuned.h5'))

y_pred = best_model.predict(X_test)
y_true_cls = np.argmax(y_test, axis=1)
y_pred_cls = np.argmax(y_pred, axis=1)

print("Test accuracy:", accuracy_score(y_true_cls, y_pred_cls))
print(multilabel_confusion_matrix(y_true_cls, y_pred_cls))

1/1 [==============================] - 1s 511ms/step
Test accuracy: 0.14285714285714285
[[[6 0]
  [1 0]]

 [[5 0]
  [2 0]]

 [[4 2]
  [0 1]]

 [[5 1]
  [1 0]]

 [[6 0]
  [1 0]]

 [[3 3]
  [1 0]]]


# 13. Visualization Utilities for Real-Time Prediction

In [14]:
def draw_styled_landmarks(image, results):
    mp_drawing.draw_landmarks(
        image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
    )
    mp_drawing.draw_landmarks(
        image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
    )
    mp_drawing.draw_landmarks(
        image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
    )

colors = [(245,117,16),(117,245,16),(16,117,245),(255,0,0),(0,255,0),(0,0,255),
          (255,255,0),(255,0,255),(0,255,255),(128,128,255),(255,128,128),
          (128,255,128),(128,128,0),(180,90,255),(90,180,255),(100,100,100)]

def prob_viz(res, actions, input_frame, colors):
    output_frame = input_frame.copy()
    for num, prob in enumerate(res):
        color = colors[num % len(colors)]
        cv2.rectangle(output_frame, (0, 60 + num * 30), (int(prob * 180), 85 + num * 30), color, -1)
        cv2.putText(output_frame, actions[num], (5, 80 + num * 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2, cv2.LINE_AA)
    return output_frame

# 14. Real-Time Inference with Hand Gating and Cooldown

In [15]:
model = load_model(os.path.join('..', 'models', 'best_wlasl_finetuned.h5'))

sequence = []
sentence = []
threshold = 0.80
cooldown_frames = 25
cooldown = 0

cap = cv2.VideoCapture(0)

with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        image, results = mediapipe_detection(frame, holistic)
        draw_styled_landmarks(image, results)

        hands_present = (results.left_hand_landmarks is not None) or (results.right_hand_landmarks is not None)

        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-SEQUENCE_LENGTH:]

        if cooldown > 0:
            cooldown -= 1

        if len(sequence) == SEQUENCE_LENGTH and hands_present and cooldown == 0:
            res = model.predict(np.expand_dims(sequence, axis=0), verbose=0)[0]
            pred_idx = np.argmax(res)
            pred_label = ACTIONS[pred_idx]
            pred_conf = res[pred_idx]

            if pred_conf > threshold and pred_label != 'Idle':
                if len(sentence) == 0 or pred_label != sentence[-1]:
                    sentence.append(pred_label)
                    cooldown = cooldown_frames

            image = prob_viz(res, ACTIONS, image, colors)

        if len(sentence) > 5:
            sentence = sentence[-5:]

        cv2.rectangle(image, (0,0), (900, 40), (245,117,16), -1)
        cv2.putText(image, ' '.join(sentence), (5,30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2, cv2.LINE_AA)

        cv2.imshow('SignMatic Realtime', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()